<a href="https://colab.research.google.com/github/njw257/bigdata/blob/main/week05_5685599_%EB%82%A8%EC%A2%85%EC%9A%B1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 과제
빅데이터프로그래밍 · 통계학과

**이름:** 남종욱

**학번:** 5685599

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week05_학번_이름.ipynb` 로 제출합니다.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)


## 문제 1. 사용자 정의 Dataset 작성 (40점)
아래 조건으로 Dataset 클래스를 작성하세요.

- 데이터 개수는 30개, 입력 특성은 4개
- `__init__` `__len__` `__getitem__` 세 메서드를 모두 구현
- 각 메서드가 하는 일을 주석으로 한 줄씩 설명


In [ ]:
# 답안
# 사용자 정의 Dataset 클래스
class MyDataset(Dataset):
    # 입력 데이터와 정답 데이터 저장
    def __init__(self, X, y):
        self.X = X
        self.y = y

    # 전체 데이터 개수 반환
    def __len__(self):
        return len(self.X)

    # 지정한 위치의 입력 데이터와 정답 반환
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# 데이터 30개, 입력 특성 4개 생성
X = torch.randn(30, 4)
y = X @ torch.tensor([[2.0], [-1.0], [0.5], [1.0]]) + 0.5

# Dataset 생성
ds = MyDataset(X, y)

# 데이터 개수 및 shape 확인
print("전체 개수:", len(ds))
print("입력 shape:", X.shape)
print("정답 shape:", y.shape)

전체 개수: 30
입력 shape: torch.Size([30, 4])
정답 shape: torch.Size([30, 1])


## 문제 2. DataLoader로 batch 출력 (30점)
문제 1의 Dataset을 DataLoader에 넣고 아래를 출력하세요.

- batch 개수
- 각 batch의 입력 shape과 정답 shape
- 마지막 batch의 크기가 다른 이유 (batch size를 7로 두고 확인)


In [ ]:
# 답안
# batch size를 7로 설정
loader = DataLoader(ds, batch_size=7, shuffle=False)

# 전체 batch 개수 출력
print("batch 개수:", len(loader))

# 각 batch의 입력 및 정답 shape 출력
for i, (bx, by) in enumerate(loader):
    print(f"batch {i+1}  X {tuple(bx.shape)}  y {tuple(by.shape)}")

batch 개수: 5
batch 1  X (7, 4)  y (7, 1)
batch 2  X (7, 4)  y (7, 1)
batch 3  X (7, 4)  y (7, 1)
batch 4  X (7, 4)  y (7, 1)
batch 5  X (2, 4)  y (2, 1)


전체 데이터 30개를 7개씩 나누면 4개의 batch에 28개가 포함되고, 남은 2개가 마지막 batch에 포함되므로 마지막 batch의 크기가 다르게 나타납니다.

## 문제 3. batch shape 설명 (10점)
문제 2의 출력에서 batch shape이 왜 그렇게 나오는지 설명하세요.
`(batch_size, 특성 수)` 의 각 숫자가 무엇을 뜻하는지 적으면 됩니다.


**답:**
batch shape에서 첫 번째 숫자는 한 번에 처리하는 데이터 개수(batch size)를, 두 번째 숫자는 각 데이터의 특성 수를 의미합니다.

따라서 입력 데이터의 shape은 (7, 4), 정답 데이터의 shape은 (7, 1)로 나타납니다. 마지막 batch는 남은 데이터가 2개이므로 각각 (2, 4), (2, 1)로 나타납니다.

## 문제 4. 학습 루프 작성 (20점)
문제 1의 Dataset으로 아래를 수행하세요.

1. `nn.Linear` 모델 하나를 만든다
2. 다섯 단계(예측 → 손실 → 초기화 → 역전파 → 수정)로 학습 루프를 작성한다
3. epoch마다 평균 손실을 출력해 손실이 내려가는지 확인한다


In [ ]:
# 답안
# 문제 1의 Dataset으로 학습용 DataLoader 생성
train_loader = DataLoader(ds, batch_size=7, shuffle=True)

# 선형 모델, 손실 함수, 옵티마이저 생성
model = nn.Linear(4, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# 학습 루프
epochs = 30

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for bx, by in train_loader:
        pred = model(bx)           # 1. 예측
        loss = loss_fn(pred, by)   # 2. 손실 계산

        optimizer.zero_grad()      # 3. 기울기 초기화
        loss.backward()           # 4. 역전파
        optimizer.step()          # 5. 파라미터 수정

        train_loss += loss.item() * bx.shape[0]

    # epoch별 평균 손실 계산 및 출력
    train_loss /= len(ds)
    print(f"epoch {epoch+1:2d}  loss {train_loss:.4f}")

epoch  1  loss 4.4680
epoch  2  loss 1.4295
epoch  3  loss 0.6110
epoch  4  loss 0.2775
epoch  5  loss 0.1381
epoch  6  loss 0.0756
epoch  7  loss 0.0426
epoch  8  loss 0.0206
epoch  9  loss 0.0104
epoch 10  loss 0.0061
epoch 11  loss 0.0031
epoch 12  loss 0.0019
epoch 13  loss 0.0011
epoch 14  loss 0.0006
epoch 15  loss 0.0003
epoch 16  loss 0.0002
epoch 17  loss 0.0001
epoch 18  loss 0.0001
epoch 19  loss 0.0000
epoch 20  loss 0.0000
epoch 21  loss 0.0000
epoch 22  loss 0.0000
epoch 23  loss 0.0000
epoch 24  loss 0.0000
epoch 25  loss 0.0000
epoch 26  loss 0.0000
epoch 27  loss 0.0000
epoch 28  loss 0.0000
epoch 29  loss 0.0000
epoch 30  loss 0.0000
